# Modelagem — Detecção de Fraude em Cartão de Crédito

CC0121 — Inteligência Artificial — UNIFAP — 2026.1

Objetivo:
1. Carregar os dados processados (de `02_preprocessing.ipynb`)
2. Balancear o treino com SMOTE (50/50)
3. Treinar Regressão Logística, Random Forest e XGBoost
4. Avaliar com Stratified K-Fold + Precision, Recall, F1, AUC-ROC
5. Comparar os modelos e salvar o melhor

**Meta da proposta:** AUC-ROC ≥ 0,95 e F1-Score (fraude) ≥ 0,80

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, f1_score, precision_score, recall_score
)

sns.set_theme(style="whitegrid")

## 2. Carregar dados processados

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

## 3. Balancear o treino com SMOTE (50/50)

Aplicamos SMOTE **somente no treino**. O teste (`X_test`, `y_test`) permanece com a distribuição real (0,17% de fraude), porque é assim que o modelo vai encontrar dados no mundo real.

In [ ]:
smote = SMOTE(sampling_strategy=1.0, random_state=42)  # 1.0 = balanceamento total (50/50)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Antes do SMOTE: ", y_train.value_counts().to_dict())
print("Depois do SMOTE:", y_train_res.value_counts().to_dict())

## 4. Função auxiliar de avaliação

Vamos usar a mesma função para avaliar os 3 modelos no conjunto de teste (real, desbalanceado).

In [ ]:
results = {}

def avaliar_modelo(nome, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    results[nome] = {"AUC-ROC": auc, "F1": f1, "Precision": precision, "Recall": recall}

    print(f"=== {nome} ===")
    print(classification_report(y_test, y_pred, target_names=["Legítima", "Fraude"]))
    print(f"AUC-ROC: {auc:.4f}\n")

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Legítima", "Fraude"]).plot(cmap="Blues")
    plt.title(f"Matriz de Confusão — {nome}")
    plt.show()

    return modelo

## 5. Modelo 1 — Regressão Logística (baseline)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_res, y_train_res)
avaliar_modelo("Regressão Logística", log_reg, X_test, y_test)

## 6. Modelo 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_res, y_train_res)
avaliar_modelo("Random Forest", rf, X_test, y_test)

## 7. Modelo 3 — XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)
xgb.fit(X_train_res, y_train_res)
avaliar_modelo("XGBoost", xgb, X_test, y_test)

## 8. Comparação dos modelos

In [ ]:
comparison_df = pd.DataFrame(results).T.sort_values("AUC-ROC", ascending=False)
comparison_df

In [ ]:
comparison_df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.title("Comparação de Modelos")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

## 9. Validação cruzada (Stratified K-Fold) do melhor modelo

Depois de escolher o melhor modelo pela comparação acima, validamos com Stratified K-Fold para confirmar que o resultado é estável e não só sorte do split.

⚠️ Ajuste a variável `melhor_modelo` abaixo para o modelo com melhor desempenho (`log_reg`, `rf` ou `xgb`).

In [ ]:
melhor_modelo = xgb  # <- ajuste aqui conforme o resultado da comparação

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    melhor_modelo,
    X_train_res, y_train_res,
    cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall"]
)

for metric in ["test_roc_auc", "test_f1", "test_precision", "test_recall"]:
    print(f"{metric}: {cv_results[metric].mean():.4f} (+/- {cv_results[metric].std():.4f})")

## 10. Salvar o melhor modelo

In [ ]:
joblib.dump(melhor_modelo, "../models/best_model.pkl")
print("Modelo salvo em models/best_model.pkl")

## 11. Checagem final contra a meta da proposta

- AUC-ROC ≥ 0,95
- F1-Score (fraude) ≥ 0,80

In [ ]:
melhor_nome = comparison_df.index[0]
auc_final = comparison_df.loc[melhor_nome, "AUC-ROC"]
f1_final = comparison_df.loc[melhor_nome, "F1"]

print(f"Melhor modelo: {melhor_nome}")
print(f"AUC-ROC: {auc_final:.4f} -> {'OK' if auc_final >= 0.95 else 'ABAIXO DA META'}")
print(f"F1 (fraude): {f1_final:.4f} -> {'OK' if f1_final >= 0.80 else 'ABAIXO DA META'}")

## 12. Próximo passo

Seguir para a construção do app Streamlit (`app/app.py`), que vai:
- Carregar `models/best_model.pkl` e `models/scaler.pkl`
- Receber os atributos de uma transação digitados pelo usuário
- Retornar a classificação (legítima/fraude) com o nível de confiança